In [ ]:
import json
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
filename = 'hopping.json'

In [ ]:
# Parse the data

with open(filename, 'r') as f:
    data = json.load(f)
df= pd.DataFrame(data)

plot_data = []
first_time=None
first_first_time = None
for i in data:
    first_time=None
    ttl = i['TTL']
    host = i['IP']
    for j in range(len(i['pings'])):
        ping = i['pings'][j]
        if first_time == None:
            first_time = ping["time"]
        if first_first_time == None:
            first_first_time = ping["time"]
        icmp_seq = ping["icmp_seq"]
        time = (ping["time"] - first_time) / 1000000000.0
        real_time = (ping["time"] - first_first_time) / 1000000000.0
        rtt = ping["RTT"]
        plot_data.append({
            "Host": host,
            "TTL": ttl,
            "RTT": rtt,
            "Seq": icmp_seq,
            "Time": time,
            "Real Time": real_time
        })
df_pings = pd.DataFrame(plot_data)
df_pings["TTL"] = df_pings["TTL"].astype("category")

In [ ]:
# Plot mean RTT by hop
palette = sns.color_palette("icefire", 3)
ax = sns.lineplot(data=df, x="TTL", y="max", marker="^", label="Max", color=palette[2])
sns.lineplot(ax=ax, data=df, x="TTL", y="mean", marker="o", label="Mean", color=palette[1])
sns.lineplot(ax=ax, data=df, x="TTL", y="min", marker="v", label="Min", color=palette[0])
plt.ylabel("RTT (ms)")
plt.xlabel("Hop")

In [ ]:
# Plot boxplot of per-hop RTT by TTL

ax = sns.boxplot(data=df_pings, x="TTL", y="RTT")
plt.ylabel("RTT (ms)")
plt.xlabel("TTL")
plt.tight_layout()
plt.show()

In [ ]:
# Plot per-hop RTT over time (by second, sequential)

ax = sns.lineplot(data=df_pings, x="Real Time", y="RTT", hue="TTL", palette="viridis", legend=True, marker="o")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="TTL")
plt.ylabel("RTT (ms)")
plt.xlabel("Time (s)")
plt.tight_layout()
# plt.ylim(0, 30)
plt.show()

In [ ]:
# Plot per-hop RTT over time (by ICMP Sequence Number, overlayed)

ax = sns.lineplot(data=df_pings, x="Seq", y="RTT", hue="TTL", palette="viridis", legend=True, marker="o")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="TTL")
plt.ylabel("RTT (ms)")
plt.xlabel("ICMP Sequence #")
plt.tight_layout()
# plt.ylim(0, 30)
plt.show()

In [ ]:
# Show TTL to IP mappings

for ttl, group in df_pings.groupby("TTL"):
    host_list = group["Host"].unique().tolist()
    print(f"{ttl}: {host_list}")